In [0]:
from pyspark.sql import functions as F

dim_date = (spark.table("workspace.silver.trips_clean")
    .select("trip_date")
    .distinct()
    .withColumn("date_key", F.date_format(F.col("trip_date"), "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("trip_date"))
    .withColumn("month", F.month("trip_date"))
    .withColumn("month_name", F.date_format("trip_date", "MMMM"))
    .withColumn("day", F.dayofmonth("trip_date"))
    .withColumn("day_of_week", F.date_format("trip_date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("trip_date").isin([1, 7]))
    .withColumn("season", 
        F.when(F.col("month").isin([12, 1, 2]), "Winter")
         .when(F.col("month").isin([3, 4, 5]), "Spring")
         .when(F.col("month").isin([6, 7, 8]), "Summer")
         .otherwise("Fall"))
)

dim_date.write.mode("overwrite").saveAsTable("workspace.gold.dim_date")

In [0]:
display(spark.sql("SELECT * FROM workspace.gold.dim_date ORDER BY trip_date LIMIT 10"))
display(spark.sql("SELECT COUNT(*) FROM workspace.gold.dim_date"))

trip_date,date_key,year,month,month_name,day,day_of_week,is_weekend,season
2025-01-01,20250101,2025,1,January,1,Wednesday,false,Winter
2025-01-02,20250102,2025,1,January,2,Thursday,false,Winter
2025-01-03,20250103,2025,1,January,3,Friday,false,Winter
2025-01-04,20250104,2025,1,January,4,Saturday,true,Winter
2025-01-05,20250105,2025,1,January,5,Sunday,true,Winter
2025-01-06,20250106,2025,1,January,6,Monday,false,Winter
2025-01-07,20250107,2025,1,January,7,Tuesday,false,Winter
2025-01-08,20250108,2025,1,January,8,Wednesday,false,Winter
2025-01-09,20250109,2025,1,January,9,Thursday,false,Winter
2025-01-10,20250110,2025,1,January,10,Friday,false,Winter


COUNT(*)
365


In [0]:
from pyspark.sql.window import Window

start_stations = (spark.table("workspace.silver.trips_clean")
    .select(F.col("start_station_name").alias("station_name")))

end_stations = (spark.table("workspace.silver.trips_clean")
    .select(F.col("end_station_name").alias("station_name")))

all_stations = start_stations.union(end_stations).filter(F.col("station_name").isNotNull())

dim_station = (all_stations
    .distinct()
    .withColumn("station_key", F.row_number().over(Window.orderBy("station_name")))
)

dim_station.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.dim_station")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(spark.sql("SELECT COUNT(*) FROM workspace.gold.dim_station"))
display(spark.sql("SELECT * FROM workspace.gold.dim_station ORDER BY station_name LIMIT 10"))

COUNT(*)
1053


station_name,station_key
1 Market St,1
1 Shortt St,2
101 Cedarvale Ave,3
11 Spadina Rd,4
111 Bond St (north Of Dundas St E) - Smart,5
111 Broadview Ave,6
12 Harbour St,7
128 Sterling Rd - Smart,8
1303 Yonge St - Smart,9
135 Queens Wharf Rd - Smart,10


In [0]:
dim_user_type = (spark.table("workspace.silver.trips_clean")
    .select("user_type")
    .distinct()
    .withColumn("user_type_key", F.row_number().over(Window.orderBy("user_type")))
)

dim_user_type.write.mode("overwrite").saveAsTable("workspace.gold.dim_user_type")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(spark.sql("SELECT * FROM workspace.gold.dim_user_type"))

user_type,user_type_key
Casual,1
Member,2


In [0]:
trips = spark.table("workspace.silver.trips_clean")
dim_date = spark.table("workspace.gold.dim_date")
dim_station = spark.table("workspace.gold.dim_station")
dim_user_type = spark.table("workspace.gold.dim_user_type")

fact_trips = (trips
    # join to dim_date on trip_date
    .join(dim_date.select("trip_date", "date_key"), on="trip_date", how="left")
    # join to dim_station for start station
    .join(dim_station.withColumnRenamed("station_key", "start_station_key"),
          trips.start_station_name == dim_station.station_name, how="left")
    .drop("station_name")
    # join to dim_station again for end station
    .join(dim_station.withColumnRenamed("station_key", "end_station_key"),
          trips.end_station_name == dim_station.station_name, how="left")
    .drop("station_name")
    # join to dim_user_type
    .join(dim_user_type, on="user_type", how="left")
    .select(
        "trip_id", "date_key", "start_station_key", "end_station_key", "user_type_key",
        "trip_duration", "start_time", "end_time", "bike_id", "bike_model",
        "temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
        "precipitation_sum", "rain_sum", "windspeed_10m_max"
    )
)

fact_trips.write.mode("overwrite").saveAsTable("workspace.gold.fact_trips")

In [0]:
display(spark.sql("SELECT COUNT(*) AS total FROM workspace.gold.fact_trips"))

display(spark.sql("""
    SELECT 
        SUM(CASE WHEN date_key IS NULL THEN 1 ELSE 0 END) AS missing_date,
        SUM(CASE WHEN start_station_key IS NULL THEN 1 ELSE 0 END) AS missing_start_station,
        SUM(CASE WHEN end_station_key IS NULL THEN 1 ELSE 0 END) AS missing_end_station,
        SUM(CASE WHEN user_type_key IS NULL THEN 1 ELSE 0 END) AS missing_user_type
    FROM workspace.gold.fact_trips
"""))

total
7807002


missing_date,missing_start_station,missing_end_station,missing_user_type
0,7,365,0


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

unknown_row = spark.createDataFrame(
    [Row(station_name="Unknown", station_key=0)],
    schema=StructType([
        StructField("station_name", StringType(), True),
        StructField("station_key", IntegerType(), True)
    ])
)

dim_station_final = dim_station.withColumn(
    "station_key", F.col("station_key").cast("int")
).unionByName(unknown_row)

dim_station_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.dim_station")

In [0]:
fact_trips_final = fact_trips.fillna({"start_station_key": 0, "end_station_key": 0})

fact_trips_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.fact_trips")

In [0]:
display(spark.sql("""
    SELECT 
        SUM(CASE WHEN start_station_key IS NULL THEN 1 ELSE 0 END) AS missing_start,
        SUM(CASE WHEN end_station_key IS NULL THEN 1 ELSE 0 END) AS missing_end
    FROM workspace.gold.fact_trips
"""))

missing_start,missing_end
0,0
